# Gold Layer Visuals

Top 5 charts from the `info_env_jordan.gold` tables (Jordan Information Environment project). Each cell queries a gold table with Spark, builds a matplotlib chart, and saves a PNG to `analysis/visualals/` in the repo for embedding in the README and publishing to GitHub.

In [0]:
# Setup: imports, output directory, shared helpers
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = "/Workspace/Users/feliciapedwards@gmail.com/Jordan Information Environment Project"
VISUALS_DIR = os.path.join(REPO_ROOT, "analysis", "visualals")
os.makedirs(VISUALS_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})


def load_gold_table(table: str, required_cols: list) -> pd.DataFrame:
    """Query a gold table and fail loudly if expected columns or rows are missing."""
    df = spark.sql(f"SELECT * FROM {table}").toPandas()
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{table}: missing expected columns {missing}; found {sorted(df.columns)}")
    if df.empty:
        raise ValueError(f"{table}: query returned 0 rows")
    return df


def to_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series)


def save_fig(fig, filename: str) -> None:
    path = os.path.join(VISUALS_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path} ({os.path.getsize(path):,} bytes)")


print("Output directory:", VISUALS_DIR)

In [0]:
# Chart 1: Weekly events vs. media tone (unified_weekly_indicators)
TABLE = "info_env_jordan.gold.unified_weekly_indicators"
df = load_gold_table(TABLE, ["week_start_date", "total_events", "avg_media_tone"])
df["week_start_date"] = to_date(df["week_start_date"])
df = df.sort_values("week_start_date")
print(f"{len(df)} weeks, {df['week_start_date'].min():%Y-%m-%d} to {df['week_start_date'].max():%Y-%m-%d}")

x = df["week_start_date"]
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)

ax1.bar(x, df["total_events"].fillna(0), width=6, alpha=0.5, color="tab:blue", label="Total events")
ax1.set_ylabel("Total events")
ax1.set_xlabel("Week")

ax2.plot(x, df["avg_media_tone"], color="tab:red", lw=1.5, label="Avg media tone")
ax2.axhline(0, color="tab:red", ls=":", lw=0.8)
ax2.set_ylabel("Avg media tone (GDELT)")
ax2.grid(False)

ax1.set_title("Weekly Events vs. Media Tone")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

save_fig(fig, "01_weekly_events_vs_media_tone.png")

In [0]:
# Chart 2: Economic protest events vs. inflation rate (theme_economic_conditions)
TABLE = "info_env_jordan.gold.theme_economic_conditions"
df = load_gold_table(
    TABLE,
    ["event_month", "inflation_protests", "fuel_price_protests", "cost_living_protests", "inflation_rate"],
)
df["event_month"] = to_date(df["event_month"])
df = df.sort_values("event_month")
print(f"{len(df)} months, {df['event_month'].min():%Y-%m} to {df['event_month'].max():%Y-%m}")

x = df["event_month"]
protest_cols = ["inflation_protests", "fuel_price_protests", "cost_living_protests"]
protests = df[protest_cols].fillna(0)

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)

bottom = pd.Series(0, index=df.index)
for col in protest_cols:
    ax1.bar(x, protests[col], width=22, bottom=bottom, alpha=0.8, label=col.replace("_", " "))
    bottom = bottom + protests[col]
ax1.set_ylabel("Protest events (stacked)")
ax1.set_xlabel("Month")

ax2.plot(x, df["inflation_rate"], color="black", lw=1.8, marker="o", ms=3, label="Inflation rate (%)")
ax2.set_ylabel("Inflation rate (%)")
ax2.grid(False)

ax1.set_title("Economic Protest Events vs. Inflation Rate")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

save_fig(fig, "02_economic_protests_vs_inflation.png")

In [0]:
# Chart 3: Labor events vs. unemployment (theme_labor_employment)
TABLE = "info_env_jordan.gold.theme_labor_employment"
df = load_gold_table(TABLE, ["event_month", "total_labor_events", "unemployment_rate", "youth_unemployment_rate"])
df["event_month"] = to_date(df["event_month"])
df = df.sort_values("event_month")
print(f"{len(df)} months, {df['event_month'].min():%Y-%m} to {df['event_month'].max():%Y-%m}")

x = df["event_month"]
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)

ax1.plot(x, df["total_labor_events"], color="tab:blue", lw=1.8, marker="o", ms=3, label="Labor events")
ax1.set_ylabel("Labor events")
ax1.set_xlabel("Month")

ax2.plot(x, df["unemployment_rate"], color="tab:orange", lw=1.5, label="Unemployment rate (%)")
ax2.plot(x, df["youth_unemployment_rate"], color="tab:red", lw=1.5, ls="--", label="Youth unemployment rate (%)")
ax2.set_ylabel("Rate (%)")
ax2.grid(False)

ax1.set_title("Labor Events vs. Unemployment")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

save_fig(fig, "03_labor_events_vs_unemployment.png")

In [0]:
# Chart 4: Economic search interest vs. weekly events (unified_weekly_indicators)
TABLE = "info_env_jordan.gold.unified_weekly_indicators"
df = load_gold_table(TABLE, ["week_start_date", "economic_search_intensity", "total_events", "demonstrations_count"])
df["week_start_date"] = to_date(df["week_start_date"])
df = df.sort_values("week_start_date")
print(f"{len(df)} weeks, {df['week_start_date'].min():%Y-%m-%d} to {df['week_start_date'].max():%Y-%m-%d}")

x = df["week_start_date"]
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)

ax1.bar(x, df["total_events"].fillna(0), width=6, alpha=0.4, color="tab:blue", label="Total events")
ax1.plot(x, df["demonstrations_count"], color="tab:green", lw=1.5, label="Demonstrations")
ax1.set_ylabel("Events per week")
ax1.set_xlabel("Week")

ax2.plot(x, df["economic_search_intensity"], color="tab:red", lw=1.8, label="Economic search intensity")
ax2.set_ylabel("Search intensity")
ax2.grid(False)

ax1.set_title("Economic Search Interest vs. Weekly Events")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

save_fig(fig, "04_search_intensity_vs_events.png")

In [0]:
# Chart 5: Distress events, search intensity & high-alert months (ml_combined_economic_themes)
TABLE = "info_env_jordan.gold.ml_combined_economic_themes"
df = load_gold_table(
    TABLE, ["event_month", "total_distress_events", "combined_search_intensity", "high_alert_flag"]
)
df["event_month"] = to_date(df["event_month"])
df = df.sort_values("event_month")

raw_flag = df["high_alert_flag"]
if raw_flag.dtype == object:
    alert = raw_flag.map(lambda v: str(v).strip().lower() in ("true", "1", "1.0"))
else:
    alert = raw_flag.fillna(0).astype(bool)
print(f"{len(df)} months, {int(alert.sum())} high-alert months")

x = df["event_month"]
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.spines["right"].set_visible(True)

colors = ["tab:red" if a else "tab:blue" for a in alert]
ax1.bar(
    x, df["total_distress_events"].fillna(0), width=22, color=colors, alpha=0.7,
    label="Distress events (red = high alert)",
)
ax1.set_ylabel("Distress events")
ax1.set_xlabel("Month")

ax2.plot(x, df["combined_search_intensity"], color="black", lw=1.8, marker="o", ms=3, label="Combined search intensity")
ax2.set_ylabel("Search intensity")
ax2.grid(False)

# Shade high-alert months
for xi, a in zip(x, alert):
    if a:
        ax1.axvspan(xi - pd.Timedelta(days=15), xi + pd.Timedelta(days=15), color="tab:red", alpha=0.08)

ax1.set_title("Distress Events, Search Intensity & High-Alert Months")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

save_fig(fig, "05_distress_composite_alerts.png")

In [0]:
# Verify all 5 PNGs exist in the repo visuals directory
expected = [
    "01_weekly_events_vs_media_tone.png",
    "02_economic_protests_vs_inflation.png",
    "03_labor_events_vs_unemployment.png",
    "04_search_intensity_vs_events.png",
    "05_distress_composite_alerts.png",
]
missing = []
for f in expected:
    path = os.path.join(VISUALS_DIR, f)
    if os.path.isfile(path) and os.path.getsize(path) > 0:
        print(f"OK       {f} ({os.path.getsize(path):,} bytes)")
    else:
        print(f"MISSING  {f}")
        missing.append(f)

assert not missing, f"Missing or empty PNGs: {missing}"
print(f"\nAll {len(expected)} charts saved to: {VISUALS_DIR}")